In [ ]:
# VERICODING ARC-AGI-3 | v16 RHAE+WASM score/pattern
# Offline mode: OPERATION_MODE=offline, ENVIRONMENTS_DIR=local
#
import subprocess, sys, os, json, time, torch, gc
import numpy as np
from pathlib import Path
from typing import Any, Optional

# ─── Offline mode for arc-agi ────────────────────────────────
os.environ.setdefault("ARC_API_KEY", "anon")
os.environ.setdefault("ARC_BASE_URL", "https://three.arcprize.org")
os.environ.setdefault("OPERATION_MODE", "offline")
os.environ.setdefault("ENVIRONMENTS_DIR",
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files")
os.environ.setdefault("RECORDINGS_DIR", "/kaggle/working/recordings")

# ─── Paths ───────────────────────────────────────────────────
DS_BASE = Path("/kaggle/input")
DS_CANDIDATES = [
    DS_BASE / "datasets/krisskey/vericoding-urm",
    DS_BASE / "vericoding-urm",
    DS_BASE / "datasets/vericoding-urm",
]
COMP_PATH = DS_BASE / "competitions/arc-prize-2026-arc-agi-3"
ENV_DIR = COMP_PATH / "environment_files"
WHEELS = COMP_PATH / "arc_agi_3_wheels"

DS = next((p for p in DS_CANDIDATES if p.is_dir() and (p / "wasm_bridge.py").exists()), None)
print(f"[v16] Dataset: {DS}")
print(f"[v16] Env dir: {ENV_DIR} (exists={ENV_DIR.is_dir()})")

if not DS:
    raise SystemExit("[v16] Dataset not found!")

sys.path.insert(0, str(DS))
for p in [DS / "external", DS / "external/urm"]:
    p.is_dir() and sys.path.insert(0, str(p))

# ─── Device detection (P100 sm_60 compatible) ───────────────
_DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        torch.zeros(1, device="cuda")
        _DEVICE = "cuda"
        _name = torch.cuda.get_device_name(0)
        _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"[v16] GPU: {_name} ({_vram:.1f} GB) — DEVICE=cuda")
    except RuntimeError as e:
        print(f"[v16] GPU unusable: {e}. DEVICE=cpu")

if _DEVICE == "cpu":
    print("[v16] CPU mode — torch.compile will be used")
else:
    print("[v16] CUDA mode — eager forward (no compile)")

# ─── Install arc-agi ─────────────────────────────────────────
try:
    import arc_agi
    print(f"[v16] arc-agi {arc_agi.__version__}")
except ImportError:
    print("[v16] Installing arc-agi...")
    pip_cmd = [sys.executable, "-m", "pip", "install", "arc-agi", "-q"]
    if WHEELS.is_dir():
        pip_cmd += ["--no-index", f"--find-links={WHEELS}"]
    subprocess.check_call(pip_cmd)
    import arc_agi

# ─── Run agent ───────────────────────────────────────────────
print(f"[v16] Running agent with TTT_STEPS=50, DEVICE={_DEVICE}...")
sys.argv = [sys.argv[0], "--train", "--ttt-steps", "50"]
from kaggle_main import main
main()